# UK Planning Scraper

Searches UK local authority planning portals for applications matching a keyword,
filtered by decision date range. Supports **Idox** and **Northgate** systems,
which cover the majority of UK councils.

**Instructions:**
1. Run *Cell 1* to install dependencies (once per Colab session)
2. Edit *Cell 3* to set your search keyword and date range
3. Run all remaining cells in order
4. Results are displayed as a table and downloaded as a CSV

> **Note:** Keywords search the **proposal/description** field only, not the address.
> A full run across all ~93 supported councils takes 30–90 minutes depending on results volume.

In [ ]:
# Cell 1: Install dependencies
!pip install requests beautifulsoup4 lxml pandas -q

In [ ]:
# Cell 2: Imports
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time
import sys
import io
import csv
import warnings
from datetime import date, datetime
from urllib.parse import urlparse, urljoin

# Many UK council sites have self-signed or misconfigured SSL certificates.
# Suppress the resulting InsecureRequestWarning so they don't clutter output.
from urllib3.exceptions import InsecureRequestWarning
warnings.filterwarnings('ignore', category=InsecureRequestWarning)

pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', 100)

print('Ready.')

In [ ]:
# Cell 3: Configuration — edit these values before running

KEYWORD      = "McDonald"           # keyword searched in proposal/description
DECIDED_FROM = date(2025, 7, 1)     # earliest decision date
DECIDED_TO   = date.today()         # latest decision date
DELAY        = 5                    # seconds between requests (please be polite)

# Optional: restrict to a subset of authorities
# Set to None to search all supported councils.
# FILTER_SYSTEM options: 'idox', 'northgate', None
# FILTER_TAG    options: 'england', 'scotland', 'wales', 'london', 'greatermanchester', etc.
FILTER_SYSTEM = None
FILTER_TAG    = None

print(f"Keyword     : {KEYWORD}")
print(f"Decided from: {DECIDED_FROM}")
print(f"Decided to  : {DECIDED_TO}")

In [ ]:
# Cell 4: Load authorities
#
# Fetches the authorities list from GitHub.
# If that fails, upload authorities.csv manually via the Colab file panel
# and set CSV_PATH to '/content/authorities.csv'.

# Points to the maintained fork with corrected URLs.
# Switch CSV_URL to the upstream master if you prefer the original:
#   "https://raw.githubusercontent.com/adrianshort/uk_planning_scraper/master/lib/uk_planning_scraper/authorities.csv"
CSV_URL  = "https://raw.githubusercontent.com/Davewest84/uk_planning_scraper/claude/understand-planning-scraper-QOyOd/lib/uk_planning_scraper/authorities.csv"
CSV_PATH = None  # override with local path if needed, e.g. '/content/authorities.csv'

def detect_system(url):
    if re.search(r'search\.do\?action=advanced', url, re.I):
        return 'idox'
    elif re.search(r'generalsearch\.aspx', url, re.I):
        return 'northgate'
    return 'unsupported'

def load_authorities(csv_url=None, csv_path=None):
    if csv_path:
        with open(csv_path) as f:
            text = f.read()
    else:
        r = requests.get(csv_url, timeout=15)
        r.raise_for_status()
        text = r.text
    reader = csv.DictReader(io.StringIO(text))
    auths = []
    for row in reader:
        system = detect_system(row['url'])
        tags = row.get('tags', '').split()
        tags.append(system)
        auths.append({
            'name':   row['authority_name'],
            'url':    row['url'],
            'system': system,
            'tags':   tags,
        })
    return auths

all_authorities = load_authorities(csv_url=CSV_URL, csv_path=CSV_PATH)

# Apply filters
authorities = [a for a in all_authorities if a['system'] in ('idox', 'northgate')]
if FILTER_SYSTEM:
    authorities = [a for a in authorities if a['system'] == FILTER_SYSTEM]
if FILTER_TAG:
    authorities = [a for a in authorities if FILTER_TAG in a['tags']]

print(f"Loaded {len(all_authorities)} total authorities")
print(f"Searching {len(authorities)} supported authorities")

In [ ]:
# Cell 5: Idox scraper
#
# Idox is used by the majority of UK councils.
# Process:
#   1. GET the advanced search page
#   2. Read all form fields (including hidden ones), override date + keyword fields
#   3. POST the form
#   4. Paginate through result list pages (each is a list of li.searchresult)
#   5. Fetch each application's detail page to get the full dataset

# Mimic a real browser to avoid WAF / Cloudflare bot-detection.
# The previous 'planning-research-bot' UA was blocked by many councils.
DEFAULT_HEADERS = {
    'User-Agent':      'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                       'AppleWebKit/537.36 (KHTML, like Gecko) '
                       'Chrome/124.0.0.0 Safari/537.36',
    'Accept':          'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Accept-Language': 'en-GB,en;q=0.9',
    'Accept-Encoding': 'gzip, deflate, br',
}

def extract_base_url(url):
    p = urlparse(url)
    return f"{p.scheme}://{p.netloc}"

def read_form_fields(soup, form_id):
    """Return (fields_dict, action_url) for a form, or (None, None) if not found."""
    form = soup.find('form', id=form_id)
    if not form:
        return None, None
    fields = {}
    for tag in form.find_all(['input', 'select', 'textarea']):
        name = tag.get('name')
        if not name:
            continue
        if tag.name == 'select':
            selected = tag.find('option', selected=True)
            fields[name] = selected['value'] if selected else ''
        elif tag.get('type') in ('checkbox', 'radio'):
            if tag.get('checked'):
                fields[name] = tag.get('value', 'on')
        else:
            fields[name] = tag.get('value', '')
    return fields, form.get('action', '')

def parse_idox_date(text):
    """Parse '12 Jan 2025' style dates from Idox pages."""
    m = re.search(r'(\d{2}\s\w{3}\s\d{4})', text)
    if m:
        try:
            return datetime.strptime(m.group(1), '%d %b %Y').date().isoformat()
        except ValueError:
            pass
    return None

IDOX_DETAIL_FIELDS = {
    'Reference':                 'council_reference',
    'Alternative Reference':     'alternative_reference',
    'Planning Portal Reference': 'alternative_reference',
    'Application Received':      'date_received',
    'Application Registered':    'date_received',
    'Application Validated':     'date_validated',
    'Address':                   'address',
    'Proposal':                  'description',
    'Status':                    'status',
    'Decision':                  'decision',
    'Decision Issued Date':      'date_decision',
    'Appeal Status':             'appeal_status',
    'Appeal Decision':           'appeal_decision',
}

def scrape_idox(authority_name, url, keyword, decided_from, decided_to, delay=5):
    base_url = extract_base_url(url)
    session  = requests.Session()
    session.headers.update(DEFAULT_HEADERS)
    # Many council SSL certificates fail verification (self-signed, wrong hostname, etc.)
    session.verify = False
    apps = []

    print(f"  GET {url}")
    try:
        resp = session.get(url, timeout=30)
        resp.raise_for_status()
    except Exception as e:
        raise RuntimeError(f"Could not load search page: {e}")

    soup = BeautifulSoup(resp.text, 'lxml')
    fields, action = read_form_fields(soup, 'searchCriteriaForm')
    if fields is None:
        # Help diagnose WHY the form wasn't found
        title = soup.find('title')
        title_text = title.get_text(strip=True) if title else '(no title)'
        h1 = soup.find('h1')
        h1_text = h1.get_text(strip=True) if h1 else ''
        hint = f'Page title: "{title_text}"'
        if h1_text:
            hint += f' | H1: "{h1_text}"'
        raise RuntimeError(f"Search form not found — {hint}")

    date_fmt = "%d/%m/%Y"
    # Set our search parameters, clear unrelated date fields
    for f in ['date(applicationReceivedStart)', 'date(applicationReceivedEnd)',
              'date(applicationValidatedStart)', 'date(applicationValidatedEnd)']:
        fields[f] = ''
    fields['date(applicationDecisionStart)'] = decided_from.strftime(date_fmt)
    fields['date(applicationDecisionEnd)']   = decided_to.strftime(date_fmt)
    fields['searchCriteria.description']     = keyword

    post_url = urljoin(base_url, action) if action else url
    try:
        resp = session.post(post_url, data=fields, timeout=30)
        resp.raise_for_status()
    except Exception as e:
        raise RuntimeError(f"Form submission failed: {e}")

    soup = BeautifulSoup(resp.text, 'lxml')
    err_block = soup.find(class_='errors')
    if err_block and re.search(r'too many results', err_block.get_text(), re.I):
        raise RuntimeError("Too many results — use a narrower date range")

    # ── Paginate through search result list pages ──────────────────────────
    page_num = 1
    while True:
        items = soup.find_all('li', class_='searchresult')
        print(f"    Page {page_num}: {len(items)} result(s)")

        for item in items:
            app = {'authority_name': authority_name,
                   'scraped_at':     datetime.now().isoformat()}

            link = item.find('a')
            app['description'] = link.get_text(strip=True) if link else ''
            app['info_url']    = urljoin(base_url, link['href']) if link else ''

            addr_tag = item.find('p', class_='address')
            app['address'] = addr_tag.get_text(strip=True) if addr_tag else ''

            meta = item.find('p', class_='metaInfo')
            if meta:
                for bit in meta.get_text().split('|'):
                    bit = bit.strip()
                    m = re.search(r'Ref\.\s*No:\s+(.+)', bit)
                    if m: app['council_reference'] = m.group(1).strip()
                    m = re.search(r'(?:Received|Registered):\s+.*?(\d{2}\s\w{3}\s\d{4})', bit)
                    if m: app['date_received'] = parse_idox_date(m.group(1))
                    m = re.search(r'Validated:\s+.*?(\d{2}\s\w{3}\s\d{4})', bit)
                    if m: app['date_validated'] = parse_idox_date(m.group(1))
                    m = re.search(r'Status:\s+(.+)', bit)
                    if m: app['status'] = m.group(1).strip()

            apps.append(app)

        next_link = soup.find('a', class_='next')
        if not next_link:
            break
        next_url = urljoin(base_url, next_link['href'])
        time.sleep(delay)
        print(f"  GET {next_url}")
        try:
            resp = session.get(next_url, timeout=30)
            resp.raise_for_status()
            soup = BeautifulSoup(resp.text, 'lxml')
            page_num += 1
        except Exception as e:
            print(f"  WARNING: Could not get next page: {e}")
            break

    # ── Fetch detail page for each application ─────────────────────────────
    print(f"  Fetching {len(apps)} detail page(s)...")
    for i, app in enumerate(apps):
        if not app.get('info_url'):
            continue
        time.sleep(delay)
        print(f"    [{i+1}/{len(apps)}] {app['info_url']}")
        try:
            resp = session.get(app['info_url'], timeout=30)
            resp.raise_for_status()
        except Exception as e:
            print(f"    WARNING: {e}")
            continue

        detail = BeautifulSoup(resp.text, 'lxml')

        # Documents count / URL
        app['documents_count'] = 0
        doc_link = detail.find(class_='associateddocument')
        if doc_link:
            a = doc_link.find('a')
        else:
            a = detail.find(id='tab_documents')
        if a:
            m = re.search(r'\d+', a.get_text())
            if m:
                app['documents_count'] = int(m.group())
                href = a.get('href', '')
                if href:
                    app['documents_url'] = urljoin(base_url, href)

        # Detail table — matched by th label, not row position
        table = detail.find(id='simpleDetailsTable')
        if table:
            for row in table.find_all('tr'):
                th = row.find('th')
                td = row.find('td')
                if not (th and td):
                    continue
                key   = th.get_text(strip=True)
                value = td.get_text(strip=True)
                if not value:
                    continue
                field = IDOX_DETAIL_FIELDS.get(key)
                if field:
                    if field.startswith('date_') and re.search(r'\d', value):
                        parsed = parse_idox_date(value)
                        app[field] = parsed if parsed else value
                    else:
                        app[field] = value

    return apps

print('Idox scraper defined.')

In [ ]:
# Cell 6: Northgate scraper
#
# Northgate is ASP.NET based. Process:
#   1. GET the search page to harvest __VIEWSTATE and __EVENTVALIDATION tokens
#   2. POST the search form with those tokens + search params
#   3. Follow the redirect to the results page (setting PS=99999 for max page size)
#   4. Parse the results table directly (no detail page needed)

def scrape_northgate(authority_name, url, keyword, decided_from, decided_to, delay=3):
    base_url    = extract_base_url(url)
    # Build the 'Generic/' base URL needed for result detail links
    generic_url = re.sub(r'(?i)[Gg]eneral[Ss]earch\.aspx.*$', 'Generic/', url)

    headers = {
        **DEFAULT_HEADERS,
        'Origin':  base_url,
        'Referer': url,
    }

    print(f"  GET {url}")
    try:
        resp = requests.get(url, headers=headers, timeout=30, verify=False)
        resp.raise_for_status()
    except Exception as e:
        raise RuntimeError(f"Could not load search page: {e}")

    soup = BeautifulSoup(resp.text, 'lxml')
    vs  = soup.find(id='__VIEWSTATE')
    ev  = soup.find(id='__EVENTVALIDATION')
    if not vs:
        raise RuntimeError("Could not find __VIEWSTATE on search page")

    cookies = {c.name: c.value for c in resp.cookies}

    form_vars = {
        'csbtnSearch':       'Search',
        'txtProposal':        keyword,
        'cboSelectDateValue': 'DATE_DECISION',
        'rbGroup':            'rbRange',
        'dateStart':          decided_from.isoformat(),  # YYYY-MM-DD
        'dateEnd':            decided_to.isoformat(),
        '__VIEWSTATE':        vs['value'],
        '__EVENTVALIDATION':  ev['value'] if ev else '',
    }

    print(f"  POST {url}")
    try:
        resp2 = requests.post(url, data=form_vars, headers=headers,
                              cookies=cookies, allow_redirects=False,
                              timeout=30, verify=False)
    except Exception as e:
        raise RuntimeError(f"Form submission failed: {e}")

    # Northgate returns 302 (Found) or occasionally 301 (Moved Permanently)
    if resp2.status_code not in (301, 302):
        raise RuntimeError(f"Expected redirect after form POST, got {resp2.status_code}")

    location     = resp2.headers.get('Location', '')
    results_path = re.sub(r'PS=\d+', 'PS=99999', location)  # request max page size
    results_url  = urljoin(base_url, results_path)

    print(f"  GET {results_url}")
    try:
        resp3 = requests.get(results_url, headers=headers, cookies=cookies,
                             timeout=30, verify=False)
        resp3.raise_for_status()
    except Exception as e:
        raise RuntimeError(f"Could not load results page: {e}")

    soup = BeautifulSoup(resp3.text, 'lxml')
    rows = soup.select('table.display_table tr')
    print(f"  Found {max(len(rows)-1, 0)} application(s)")

    apps = []
    for row in rows:
        cells = row.find_all('td')
        if not cells:  # header row (only th's)
            continue

        app = {'authority_name': authority_name,
               'scraped_at':     datetime.now().isoformat()}

        app['council_reference'] = cells[0].get_text(strip=True)

        link = cells[0].find('a')
        if link:
            href = re.sub(r'[\x00-\x1f]', '', link.get('href', ''))  # strip junk chars
            app['info_url'] = urljoin(generic_url, href)

        app['address']     = cells[1].get_text(strip=True) if len(cells) > 1 else ''
        app['description'] = cells[2].get_text(strip=True) if len(cells) > 2 else ''
        app['status']      = cells[3].get_text(strip=True) if len(cells) > 3 else ''

        raw_validated = cells[4].get_text(strip=True) if len(cells) > 4 else ''
        if raw_validated and raw_validated != '--':
            try:
                app['date_validated'] = datetime.strptime(raw_validated, '%d-%m-%Y').date().isoformat()
            except ValueError:
                app['date_validated'] = raw_validated

        # Some Northgate councils omit the decision column
        app['decision'] = cells[5].get_text(strip=True) if len(cells) > 5 else ''

        apps.append(app)

    return apps

print('Northgate scraper defined.')

In [ ]:
# Cell 7: Run the search
#
# Iterates through all selected authorities, scrapes each one,
# and collects results. Errors for individual councils are caught
# and logged so the run continues.
#
# TIP: If you only want to test a few councils first, change
#      'authorities' to e.g. 'authorities[:5]' below.

all_results = []
all_errors  = []

print(f"Searching {len(authorities)} authorities")
print(f"Keyword: '{KEYWORD}'  |  Decided: {DECIDED_FROM} → {DECIDED_TO}")
print('=' * 70)

for i, auth in enumerate(authorities):  # change to authorities[:5] to test a subset
    print(f"\n[{i+1}/{len(authorities)}] {auth['name']} ({auth['system']})")
    sys.stdout.flush()

    try:
        if auth['system'] == 'idox':
            apps = scrape_idox(
                auth['name'], auth['url'],
                KEYWORD, DECIDED_FROM, DECIDED_TO,
                delay=DELAY,
            )
        else:
            apps = scrape_northgate(
                auth['name'], auth['url'],
                KEYWORD, DECIDED_FROM, DECIDED_TO,
                delay=DELAY,
            )

        if apps:
            print(f"  ✓ Found {len(apps)} application(s)")
            for app in apps:
                ref  = app.get('council_reference', '')
                addr = app.get('address', '')[:70]
                print(f"    {ref} | {addr}")
            all_results.extend(apps)
        else:
            print("  → No results")

    except Exception as e:
        msg = f"{type(e).__name__}: {e}"
        print(f"  ERROR: {msg}")
        all_errors.append({
            'authority': auth['name'],
            'system':    auth['system'],
            'error':     msg,
        })

print(f"\n{'='*70}")
print(f"COMPLETE — {len(all_results)} application(s) found")
print(f"Authorities with errors: {len(all_errors)}")

In [ ]:
# Cell 8: Display results and download as CSV

COLUMN_ORDER = [
    'authority_name', 'council_reference', 'date_decision',
    'address', 'description', 'status', 'decision',
    'date_received', 'date_validated',
    'info_url', 'documents_count', 'documents_url',
    'alternative_reference', 'appeal_status', 'appeal_decision',
    'scraped_at',
]

if all_results:
    df = pd.DataFrame(all_results)
    df = df.reindex(columns=[c for c in COLUMN_ORDER if c in df.columns])
    display(df)

    csv_filename = f"planning_{KEYWORD.lower()}_{DECIDED_FROM}_{DECIDED_TO}.csv"
    df.to_csv(csv_filename, index=False)
    print(f"\nSaved {len(df)} row(s) to {csv_filename}")

    try:
        from google.colab import files
        files.download(csv_filename)
    except ImportError:
        print(f"(Not running in Colab — find the CSV in your working directory)")
else:
    print("No results found.")

# Show error summary
if all_errors:
    print(f"\n--- Authorities that errored ({len(all_errors)}) ---")
    err_df = pd.DataFrame(all_errors)
    display(err_df)